# 6c · Single-call end-to-end reified extraction  (BASELINE)

The **end-to-end baseline** for the staged event-centric pipeline (nb2→nb4→nb6b). Instead of extracting
entities, typing/deduping them, and *then* extracting n-ary nodes + relations over that frozen entity
set, this notebook does it all in **ONE Gemini call from the raw TEXT**: entities (with open free-form
types), n-ary nodes (Event / Observation / Claim), and open `relations`.

It exists to answer *"is the staged decomposition actually better than doing everything in one call?"* —
run it alongside the staged path and compare (context completeness, fragmentation, predicate/type
economy, QA faithfulness, entity precision/recall, truncation rate).

Output is the **same shape** as nb6b's `nary_graph.json`, written to `results/<id>/nary_graph_e2e.json`,
so it flows through the SAME downstream:
- **nb7b** canonicalizes its relations — add `"nary_graph_e2e.json"` to nb7b's input glob/name.
- **nb8** renders it — add `"nary_graph_e2e.json"` to `REL_FILES`.

(Entity *types* here are open too; canonicalizing them is the nb3/nb4-style step — the same gap the
staged n-ary path has. This notebook is the extraction baseline; canonicalization is shared downstream.)

In [172]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [173]:
# --- Config -------------------------------------------------------------------------------
from pathlib import Path

RESULTS_DIR  = ROOT / "results"            # writes results/<id>/nary_graph_e2e.json
DATASET_DIR  = ROOT / "dataset/articles"   # reads <id>.txt (raw text — NO pre-extracted entities)
GEMINI_MODEL = "gemini-2.5-flash"

ARTICLE_ID   = "art5"                      # single article used by the demo cell below
SELECTED_ARTICLES = ["art1", "art2", "art4", "art5"]   # batch set

OUTPUT_NAME  = "e2e/nary_graph_dedup.json"       # written to results/<id>/  (distinct from the staged nary_graph.json)

# Articles this run processes: those with a raw <id>.txt.
_available = sorted(p.stem for p in DATASET_DIR.glob("*.txt"))
if SELECTED_ARTICLES is None:
    ARTICLE_IDS = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    assert not missing, f"no article text for: {missing}"
    ARTICLE_IDS = list(SELECTED_ARTICLES)

print(f"{len(_available)} article texts available; this run will process {ARTICLE_IDS}")

18 article texts available; this run will process ['art1', 'art2', 'art4', 'art5']


In [174]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time, re
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None

MAX_OUTPUT_TOKENS = None   # None -> no cap (model max). This call is BIG (entities+nodes+relations at once).
THINKING_BUDGET   = 0      # thinking OFF so the whole budget goes to the JSON (avoids truncation)

def _loads_lenient(text, resp=None):
    t = (text or "").strip()
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z]*\n?", "", t); t = re.sub(r"\n?```$", "", t).strip()
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        pass
    t2 = re.sub(r",(\s*[}\]])", r"\1", t)
    try:
        return json.loads(t2)
    except json.JSONDecodeError as e:
        fr = None
        try: fr = resp.candidates[0].finish_reason
        except Exception: pass
        raise ValueError(f"Gemini JSON unparseable (finish_reason={fr}; {len(t)} chars). If MAX_TOKENS, "
                         f"the single call TRUNCATED — this is expected to happen sooner than the staged path "
                         f"(bigger output). json error: {e}\n...tail:\n{t[-300:]}") from None

def gemini_json(system_prompt, user_message, model=None, temperature=0):
    global last_latency_s
    cfg = dict(system_instruction=system_prompt, temperature=temperature, response_mime_type="application/json")
    if MAX_OUTPUT_TOKENS is not None:
        cfg["max_output_tokens"] = MAX_OUTPUT_TOKENS
    if THINKING_BUDGET is not None and hasattr(genai_types, "ThinkingConfig"):
        cfg["thinking_config"] = genai_types.ThinkingConfig(thinking_budget=THINKING_BUDGET)
    t0 = time.perf_counter()
    resp = client.models.generate_content(model=model or GEMINI_MODEL, contents=user_message,
                                          config=genai_types.GenerateContentConfig(**cfg))
    last_latency_s = time.perf_counter() - t0
    return _loads_lenient(resp.text, resp)

python(93879) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Note: you may need to restart the kernel to use updated packages.


## 1. Single-call extraction — entities + n-ary nodes + relations, in one pass

One Gemini call takes the raw TEXT and returns everything: entities (with open free-form types and
coreference merged), n-ary nodes (Event / Observation / Claim), and open `relations`. The rules are the
same event-centric rules as nb6b, plus entity extraction and coreference in the same pass — which is
exactly the burden this baseline puts on a single call.

In [175]:
# E2E_SYSTEM_PROMPT = """You are an expert knowledge-graph extraction system. Given a news TEXT and its REFERENCE_DATE (ISO-8601), build a reified, event-centric knowledge graph in ONE pass, with THREE parts.

# (1) ENTITIES — the real-world things the TEXT mentions (people, organizations, places, quantities, dates, durations, etc.). Each entity: {"id" ("E1","E2",...), "text" (canonical surface form), "type" (a concise free-form PascalCase class label, as specific as the TEXT warrants, but a REUSABLE class with no specific named entity in the label), "aliases" (other surface forms for the SAME entity), "normalized_value", "normalized_end" (ISO-8601 for dates/times per the TEMPORAL rule; null otherwise), "confidence" (0-1, your certainty in the type)}. MERGE coreferent mentions into ONE entity; never duplicate an entity. Keep type labels consistent: the same kind of thing gets the exact same label.

# (2) N-ARY NODES — new reified nodes for the events, observations, and claims the TEXT is about. Kinds:
# - "Event": something that happens or is scheduled/forecast to happen.
# - "Observation": a stated measurement, statistic, quantity, or quantified change.
# - "Claim": a statement explicitly attributed to a source.
# Each n-ary node: {"id" ("EV1"/"OBS1"/"CL1",...), "node_kind", "type" (concise free-form PascalCase; reusable, no named entity in the label), "label" (a SHORT natural title for the event/observation/claim - a few words, no packed-in facts), "is_main" (bool), "evidence" (verbatim span(s)), "confidence" (0-1)}.

# (3) RELATIONS — open, free-form directed relations {"subject_id","relation_phrase","object_id","evidence","confidence"} where subject_id and object_id are ids of ENTITIES or N-ARY NODES. Choose a concise, GENERAL, reusable relation_phrase: prefer a short present-tense verb phrase (e.g. "located in", "attributed to", "caused by") that names the relation itself (canonicalized downstream) WITHOUT baking the object/subject into it; keep phrasing consistent (the same relation gets the same phrase) and orient each triple so subject --relation_phrase--> object reads correctly. Define each unique phrase in relation_definitions.

# RULES:
# 1. Identify the article's MAIN event(s) and set "is_main": true on them.
# 2. Tie every participant of a stated fact to its n-ary node with a relation, and capture the stable entity-to-entity relations. Aim for ONE connected graph — do not leave a relevant node dangling.
# 3. Keep every fact's context connected to it: attach all the details that qualify a fact — value, time, place, cause, qualifier, source — to THAT fact's n-ary node, so it forms one connected cluster reachable in a few hops.
# 4. Keep each n-ary node's label a SHORT, natural title (a few words, ~5 max) that reads as the event/observation/claim; do NOT pack facts into it - values, times, quantities, sources and participants are RELATIONS, not label text. Any entity you name in the label MUST also be attached as a relation (the label never replaces an edge).
# 5. Give every Observation and Claim at least one relation to the Event it describes, and relate each Event to other Events it is linked to in the TEXT if applicable (e.g. "part of" / "caused by" / "led to" / "followed").
# 6. When a breakdown gives several values that each belong to a distinct item (category, group, period, round), relate EACH value to its item AND link each item to the shared parent fact/event, so the whole breakdown stays in ONE connected cluster.
# 7. subject_id and object_id MUST be ids you defined in ENTITIES or N-ARY NODES; never relate a node to itself.
# 8. Extract every relevant entity; omit only boilerplate (navigation, "related"/"read more" links).
# 9. Extract only what the TEXT explicitly states or unambiguously entails; do NOT use outside knowledge. "evidence" is a list of VERBATIM, UNBROKEN spans copied character-for-character from the TEXT — never abbreviate, paraphrase, normalize punctuation, or join distant fragments with "…". Together the spans MUST contain the subject, the object, and the wording that expresses the relation (for an n-ary node, the wording that states the event/observation/claim). When a single sentence contains all of them, give that one sentence as the only span; when they are stated across different sentences, give EACH relevant sentence as its own separate span in the list rather than splicing them into one elided quote. Prefer the smallest set of COMPLETE sentences that fully supports the assertion. "confidence" (0-1) is lower when a fact, relation, or direction is implicit or ambiguous.
# 10. Normalize temporal entities: for any entity that denotes a date/time (absolute like "24 June", or relative/deictic like "Wednesday" / "the last week of the month" / "this weekend"), resolve it to an ISO-8601 "normalized_value" using REFERENCE_DATE and the narrative direction, at the granularity the text justifies; for a span also set "normalized_end" (inclusive end); for non-temporal or unresolvable entities set both to null. Never put the normalized value in "text".
# 11. Define every unique label: type_definitions: for each distinct entity "type", a general one-sentence "definition" + 2-5 "synonyms" (independent of the specific entities). relation_definitions: for each "relation_phrase", a NEUTRAL one-sentence "definition" (do not name subject/object or state a direction, so it reads the same for the relation and its inverse) + 2-5 "synonyms".

# OUTPUT strict JSON:
# {
#   "entities": [ {"id","text","type","aliases":[...],"normalized_value","normalized_end","confidence"} ],
#   "nary_nodes": [ {"id","node_kind","type","label","is_main","evidence":[...],"confidence"} ],
#   "relations": [ {"subject_id","relation_phrase","object_id","evidence":[...],"confidence"} ],
#   "type_definitions": [ {"type","definition","synonyms":[...]} ],
#   "relation_definitions": [ {"relation_phrase","definition","synonyms":[...]} ]
# }
# """

def _demojibake(s):
    if isinstance(s, str) and any(m in s for m in ("Ã", "Â", "â€")):
        try: return s.encode("latin-1").decode("utf-8")
        except (UnicodeDecodeError, UnicodeEncodeError): return s
    return s

def _reference_date(text):
    m = re.search(r"^Date:\s*(\d{2})/(\d{2})/(\d{4})", text, re.MULTILINE)
    if not m: return None
    d, mth, y = m.groups(); return f"{y}-{mth}-{d}"

def extract_e2e(aid, write=True):
    """ONE Gemini call: raw TEXT -> entities + n-ary nodes + relations -> results/<id>/nary_graph_e2e.json."""
    text = _demojibake((DATASET_DIR / f"{aid}.txt").read_text())
    ref_date = _reference_date(text)
    out = gemini_json(E2E_SYSTEM_PROMPT, f"REFERENCE_DATE: {ref_date}\n\nTEXT:\n{text}")

    entities   = out.get("entities", [])
    nary_nodes = out.get("nary_nodes", [])
    relations  = out.get("relations", [])
    for e in entities:                                  # normalise: ensure aliases list exists
        e.setdefault("aliases", [])
    node_text = {e["id"]: (e.get("text"), e.get("type")) for e in entities}
    node_text.update({n["id"]: (n.get("label"), n.get("type")) for n in nary_nodes})
    for r in relations:                                 # resolve endpoints (so nb7b/nb8 read subject/object surface+type)
        r["subject_text"], r["subject_type"] = node_text.get(r.get("subject_id"), (None, None))
        r["object_text"], r["object_type"]   = node_text.get(r.get("object_id"), (None, None))

    graph = {"article_id": aid, "reference_date": ref_date, "method": "single_call_e2e",
             "entities": entities, "nary_nodes": nary_nodes,
             "relations": relations, "type_definitions": out.get("type_definitions", []),
             "relation_definitions": out.get("relation_definitions", [])}

    if REVIEW:                                          # optional 2nd LLM pass: check/repair vs TEXT
        graph = review_graph(aid, text, graph, ref_date)
        entities, nary_nodes, relations = graph["entities"], graph["nary_nodes"], graph["relations"]

    if write:
        p = RESULTS_DIR / aid; p.mkdir(parents=True, exist_ok=True)
        (p / OUTPUT_NAME).write_text(json.dumps(graph, indent=2, ensure_ascii=False))
        print(f"  wrote {p / OUTPUT_NAME}  ({len(entities)} entities, {len(nary_nodes)} n-ary nodes, "
              f"{len(relations)} relations)  [{last_latency_s:.1f}s]")
    return graph

In [176]:
# # --- EXPERIMENTAL alternative prompt: HOLISTIC (not decomposed into 3 separate tasks) ---
# # Same rules + same OUTPUT schema as the 3-part prompt above (so extract_e2e / 6d / 6e / nb8 all still
# # work), but framed as building ONE reified, event-centric KG directly instead of "extract entities,
# # THEN n-ary nodes, THEN relations". Run THIS cell to use it; re-run the cell above to go back to the
# # 3-part prompt. (extract_e2e reads the global E2E_SYSTEM_PROMPT at call time.)
# # NOTE: outputs still overwrite the same OUTPUT_NAME (nary_graph_e2e.json). To keep both variants,
# # set OUTPUT_NAME (in the config cell) to e.g. "e2e/nary_graph_holistic.json" before running the batch.
# E2E_SYSTEM_PROMPT_HOLISTIC = """You are an expert knowledge-graph extraction system. Given a news TEXT and its REFERENCE_DATE (ISO-8601), represent everything the TEXT states as ONE reified, event-centric knowledge graph, built directly in a single pass.

# In this graph the news FACTS are first-class reified nodes and the real-world entities are wired to them:
# - Reified fact nodes (n-ary nodes) stand for the events, observations, and claims the TEXT is about \u2014 an "Event" (something that happens or is scheduled/forecast to happen), an "Observation" (a stated measurement, statistic, quantity, or quantified change), or a "Claim" (a statement explicitly attributed to a source). Each has a SHORT natural title (a few words, ~5 max) that reads as the event/observation/claim, with no packed-in facts - values, times, sources and participants are relations, not label text; any entity named in a label must also be attached as a relation.
# - Entities are the people, organizations, places, quantities, dates, durations, etc. Coreferent mentions are MERGED into ONE entity, never duplicated.
# - Relations are open, free-form directed edges (invent the concise, GENERAL, reusable relation_phrase that names the relation itself (without baking the object/subject into it) \u2014 canonicalized downstream) between any two ids, entity or reified node, that connect every fact to its participants and context and connect facts to one another, so the whole graph is ONE connected component.

# Build the graph so it satisfies:
# 1. The article's MAIN fact(s) carry "is_main": true.
# 2. Every participant and every qualifying detail of a stated fact \u2014 value, time, place, cause, qualifier, source \u2014 attaches to THAT fact's reified node as a relation, forming one connected cluster reachable in a few hops; also capture the stable entity-to-entity relations, and leave no relevant node dangling.
# 3. Keep each reified node's label a SHORT handle; express values, places, times, causes and participants as RELATIONS to entities, not inside the label.
# 4. Every Observation and Claim has at least one relation to the Event it describes; relate each Event to other Events it is linked to in the TEXT (e.g. "part of" / "caused by" / "led to" / "followed").
# 5. When a breakdown gives several values that each belong to a distinct item (category, group, period), relate EACH value directly to the item it belongs to (value -> item).
# 6. Every subject_id/object_id MUST be an id you defined; never relate a node to itself.
# 7. Include every relevant entity; omit only boilerplate (navigation, "related"/"read more" links).
# 8. Extract only what the TEXT explicitly states or unambiguously entails; do NOT use outside knowledge. "evidence" is a list of VERBATIM, UNBROKEN spans copied character-for-character from the TEXT — never abbreviate, paraphrase, normalize punctuation, or join distant fragments with "…". Together the spans MUST contain the subject, the object, and the wording that expresses the relation (for an n-ary node, the wording that states the event/observation/claim). When a single sentence contains all of them, give that one sentence as the only span; when they are stated across different sentences, give EACH relevant sentence as its own separate span in the list rather than splicing them into one elided quote. Prefer the smallest set of COMPLETE sentences that fully supports the assertion. "confidence" (0-1) is lower when a fact, relation, or direction is implicit or ambiguous.
# - Temporal normalization: for any entity that denotes a date/time (absolute like "24 June", or relative/deictic like "Wednesday" / "the last week of the month" / "this weekend"), resolve it to an ISO-8601 "normalized_value" using REFERENCE_DATE and the narrative direction, at the granularity the text justifies; for a span also set "normalized_end" (inclusive end); for non-temporal or unresolvable entities set both to null. Never put the normalized value in "text".
# - Definitions: after extraction, define every unique label. type_definitions: for each distinct entity "type", a general one-sentence "definition" + 2-5 "synonyms" (independent of the specific entities). relation_definitions: for each "relation_phrase", a NEUTRAL one-sentence "definition" (do not name subject/object or state a direction, so it reads the same for the relation and its inverse) + 2-5 "synonyms".

# Serialize the graph as strict JSON with these collections:
# {
#   "entities": [ {"id" ("E1",...), "text" (canonical surface form), "type" (concise free-form PascalCase class; reusable, no named entity in the label), "aliases":[...], "normalized_value", "normalized_end", "confidence" (0-1)} ],
#   "nary_nodes": [ {"id" ("EV1"/"OBS1"/"CL1",...), "node_kind", "type" (concise free-form PascalCase; reusable, no named entity in the label), "label" (SHORT natural title; no packed-in facts), "is_main" (bool), "evidence":[verbatim span(s)], "confidence" (0-1)} ],
#   "relations": [ {"subject_id","relation_phrase","object_id","evidence":[...],"confidence"} ],
#   "type_definitions": [ {"type","definition","synonyms":[...]} ],
#   "relation_definitions": [ {"relation_phrase","definition","synonyms":[...]} ]
# }
# """

# # Activate the holistic prompt (comment out the next line to keep the 3-part prompt above active):
# E2E_SYSTEM_PROMPT = E2E_SYSTEM_PROMPT_HOLISTIC
# print("E2E prompt: HOLISTIC (single reified-KG instruction) is now ACTIVE  \u2014 re-run the previous cell to revert to the 3-part prompt")


In [177]:
# --- EXPERIMENTAL prompt v4: 3-PART + DEDUP + aboutness / anti-over-reification ---
# Builds on the 3-part dedup prompt. Reorganized for readability (bulleted field schemas + grouped
# rules). Rules ADDED (these change behavior vs the earlier dedup prompt):
#   * Aboutness: every n-ary node — and each main Event — must link to the core subject/topic entity.
#   * Reification discipline: prefer a direct entity-to-entity relation over a Claim/Observation when it
#     is clearer/more logical (don't over-reify stable, definitional facts).
#   * Entity materialization: if a relation needs a subject/object not yet in ENTITIES (incl. anything
#     named only inside an n-ary label), create that entity and use its id.
#   * Final review/repair is now a SEPARATE 2nd LLM call (review_graph below) — NOT part of this prompt.
# Run THIS cell to use it; re-run the 3-part cell (top) or the holistic cell to switch.
# NOTE: still writes OUTPUT_NAME. Set OUTPUT_NAME (config cell) to e.g. "e2e/nary_graph_dedup.json"
# before the batch to keep this variant separate.
E2E_SYSTEM_PROMPT_DEDUP = """You are an expert knowledge-graph extraction system. Given a news TEXT and its REFERENCE_DATE (ISO-8601), build a reified, event-centric knowledge graph in one pass, with three parts.

PART 1 - ENTITIES
The real-world things the TEXT mentions. Each entity:
- "id": "E1","E2",...
- "text": canonical surface form
- "type": a concise free-form PascalCase class label, as specific as the TEXT warrants, but a reusable class with no specific named entity in the label
- "aliases": other surface forms for the same entity
- "normalized_value" / "normalized_end": ISO-8601 for dates/times per the Temporal normalization rule; null otherwise
- "confidence": 0-1, your certainty in the type

Merge coreferent mentions into one entity; never duplicate an entity. Keep type labels consistent: the same kind of thing gets the exact same label.

PART 2 - N-ARY NODES
New reified nodes for the events and claims the TEXT is about. Kinds:
- "Event": something that happens or is scheduled/forecast to happen.
- "Claim": a statement explicitly attributed to a source.

Each n-ary node:
- "id": "EV1"/"OBS1"/"CL1",...
- "node_kind": Event or Claim
- "type": concise free-form PascalCase; reusable, no named entity in the label
- "label": a short, natural title
- "is_main": bool
- "evidence": verbatim span(s)
- "confidence": 0-1

PART 3 - RELATIONS
Open, free-form directed relations. Each relation:
- "subject_id" / "object_id": ids of ENTITIES or N-ARY NODES
- "relation_phrase": concise, general, and reusable; without baking the object/subject into it; orient each triple so subject --relation--> object reads correctly.
- "evidence" / "confidence"

RULES:

Main event & event structure:
- Identify the article's main event(s) and set "is_main": true on them.
- Give every Claim at least one relation to the Event it is related to, and relate each Event to other Events it is linked to in the TEXT if applicable (e.g. "part of" / "caused by" / "led to" / "followed").

Aboutness:
- Every n-ary node must link to the core subject/topic entity it concerns via an "about" relation. In particular, connect each main Event to the primary entity the article centres on (not an n-ary node), so that central entity is not left stranded.

Reification discipline (do not over-reify):
- When a fact is a stable, direct relationship between two entities and a direct entity-to-entity relation expresses it clearly and logically, use the direct relation instead of creating an n-ary node. Reserve Event/Claim nodes for genuine events and attributed statements.

Deduplication:
- Deduplicate n-ary nodes: never emit two nodes for the same event, or claim. When a fact appears more than once (restated, elaborated, described from another angle, or attributed again), keep a single surviving node and fold every connected entity, value, time, place, cause, qualifier, source and relation from all the duplicate mentions onto that one node, unioning their evidence spans — so no participant or detail is lost. This also applies across node kinds: if an event/claim you reified as an n-ary node would also appear as a standalone entity, do NOT create that separate ENTITY — use the n-ary node's id as the relation argument (if one is created anyway, link it "sameAs" the node).

Connectivity & context:
- Tie every participant of a stated fact to its n-ary node with a relation, and capture all the the entity-to-entity relations mentioned in text. Aim for one connected graph — do not leave a relevant node dangling.
- Every subject_id and object_id MUST resolve to an id that exists in ENTITIES or N-ARY NODES.

Labels:
- Keep each n-ary node's label a short, natural title (a few words, ~5 max) that reads as the event/claim; do NOT pack facts into it - values, times, quantities, sources and participants are RELATIONS, not label text. Any entity you name in the label MUST also be attached as a relation (the label never replaces an edge).

Value breakdowns:
- When a breakdown gives several values that each belong to a distinct item (a category, group, period, rank, etc), attach each value directly to its own item (item --relation--> value), then link each item to the shared parent fact/event. Do NOT attach both the values and the items directly to the parent as separate edges — that loses which value pairs with which item (i.e. item --relation--> its own value, NOT parent --relation--> every value alongside parent --relation--> every item).

Coverage & faithfulness:
- Extract every relevant entity; omit only boilerplate (navigation, "related"/"read more" links).
- Extract only what the TEXT explicitly states or unambiguously entails; do NOT use outside knowledge. 
- "evidence" is a list of VERBATIM, UNBROKEN spans copied character-for-character from the TEXT — never abbreviate, paraphrase, normalize punctuation, or join distant fragments with "…". Together the spans MUST contain the subject, the object, and the wording that expresses the relation (for an n-ary node, the wording that states the event/claim). When a single sentence contains all of them, give that one sentence as the only span; when they are stated across different sentences, give EACH relevant sentence as its own separate span in the list rather than splicing them into one elided quote. Prefer the smallest set of COMPLETE sentences that fully supports the assertion. 
- "confidence" (0-1) is lower when a fact, relation, or direction is implicit or ambiguous.

Temporal normalization:
- Normalize temporal entities: for any entity that denotes a date/time (absolute like "24 June", or relative/deictic like "Wednesday" / "the last week of the month" / "this weekend"), resolve it to an ISO-8601 "normalized_value" using REFERENCE_DATE and the narrative direction, at the granularity the text justifies; for a span also set "normalized_end" (inclusive end); for non-temporal or unresolvable entities set both to null. Never put the normalized value in "text".

Definitions:
- Define every unique label: type_definitions: for each distinct entity "type", a general one-sentence "definition" + 2-5 "synonyms" (independent of the specific entities). relation_definitions: for each "relation_phrase", a neutral one-sentence "definition" (do not name subject/object or state a direction, so it reads the same for the relation and its inverse) + 2-5 "synonyms".

OUTPUT strict JSON:
{
  "entities": [ {"id","text","type","aliases":[...],"normalized_value","normalized_end","confidence"} ],
  "nary_nodes": [ {"id","node_kind","type","label","is_main","evidence":[...],"confidence"} ],
  "relations": [ {"subject_id","relation_phrase","object_id","evidence":[...],"confidence"} ],
  "type_definitions": [ {"type","definition","synonyms":[...]} ],
  "relation_definitions": [ {"relation_phrase","definition","synonyms":[...]} ]
}
"""

#- Keep every fact's context connected to it: attach all the details that qualify a fact — value, time, place, cause, qualifier, source — to that fact's n-ary node, so it forms one connected cluster reachable in a few hops.


# Activate the dedup prompt (comment out to keep whichever prompt is currently active):
E2E_SYSTEM_PROMPT = E2E_SYSTEM_PROMPT_DEDUP
print("E2E prompt: 3-PART + DEDUP + aboutness/anti-over-reify is now ACTIVE  — re-run the 3-part or holistic cell to switch")

E2E prompt: 3-PART + DEDUP + aboutness/anti-over-reify is now ACTIVE  — re-run the 3-part or holistic cell to switch


In [178]:
# ── OPTIONAL review/repair pass (SEPARATE 2nd LLM call), replaces the old in-prompt PART 4 ───────────
# Takes the TEXT + the draft graph and returns a small PATCH (not a full rewrite): it materialises any
# referenced-but-missing endpoint entity, adds relations the TEXT states but the draft missed, collapses
# an over-reified n-ary node into a direct entity-to-entity edge, and drops redundant/incorrect edges.
# Returning a PATCH (rather than the whole graph) means the pass can only touch what it names — it cannot
# silently drop the rest. A deterministic guard then removes any relation whose endpoint is still
# undefined, so the written graph is always self-consistent. Set REVIEW=False to skip the pass.
REVIEW = True

REVIEW_PROMPT = """You are an expert knowledge-graph reviewer and repairer. You are given a news TEXT and a DRAFT graph (entities, nary_nodes, relations) already extracted from it. Do NOT rebuild the graph. Inspect it against the TEXT and return a small PATCH that fixes ONLY these problems:

1. MISSING ENTITIES — every subject_id / object_id used by a relation MUST exist in entities or nary_nodes. If a relation references an id that is not defined (or a stated fact clearly needs a participant that was never created — e.g. a measured value, a place, an instrument), add that entity with its surface text and a concise reusable PascalCase type taken from the TEXT. Reuse the exact referenced id when materialising a referenced-but-missing id; otherwise pick a fresh unused id.
2. MISSING RELATIONS — add any relation the TEXT states or unambiguously entails that the draft omitted (entity-to-entity, or entity-to-n-ary-node). Never invent a relation the TEXT does not support.
3. OVER-REIFICATION — if an n-ary node encodes what is really a stable, direct fact between two entities, collapse it: list the node id in remove_nary_nodes and re-express the fact as a direct entity-to-entity relation in add_relations (carry its evidence). Only when no information is lost.
4. REDUNDANT / WRONG edges — remove exact-duplicate edges, and fix a wrong direction (remove the wrong one and add the corrected one).

Keep every existing correct fact; do not remove anything unless it is redundant or you are re-expressing it. Return ONLY valid JSON — a PATCH, using empty lists where there is nothing to change:
{
  "add_entities":      [ {"id","text","type","aliases":[],"normalized_value":null,"normalized_end":null,"confidence":1.0} ],
  "add_nary_nodes":    [ {"id","node_kind","type","label","is_main":false,"evidence":[],"confidence":1.0} ],
  "remove_nary_nodes": [ "<id>" ],
  "add_relations":     [ {"subject_id","relation_phrase","object_id","evidence":[],"confidence":1.0} ],
  "remove_relations":  [ {"subject_id","relation_phrase","object_id"} ],
  "add_relation_definitions": [ {"relation_phrase","definition","synonyms":[]} ],
  "add_type_definitions":     [ {"type","definition","synonyms":[]} ],
  "notes": "one short line on what you changed"
}"""

def _resolve_endpoint_texts(graph):
    """Recompute subject/object text+type on every relation from the current entities + n-ary nodes."""
    ref = {e["id"]: (e.get("text"), e.get("type")) for e in graph["entities"]}
    ref.update({n["id"]: (n.get("label"), n.get("type")) for n in graph["nary_nodes"]})
    for r in graph["relations"]:
        r["subject_text"], r["subject_type"] = ref.get(r.get("subject_id"), (None, None))
        r["object_text"],  r["object_type"]  = ref.get(r.get("object_id"),  (None, None))
    return graph

def _guard_endpoints(graph):
    """Drop any relation whose subject or object id is still undefined (last-resort self-consistency)."""
    ids = {e["id"] for e in graph["entities"]} | {n["id"] for n in graph["nary_nodes"]}
    kept, dropped = [], []
    for r in graph["relations"]:
        (kept if r.get("subject_id") in ids and r.get("object_id") in ids else dropped).append(r)
    graph["relations"] = kept
    return graph, dropped

def _apply_patch(graph, patch):
    ents  = {e["id"]: e for e in graph["entities"]}
    nodes = {n["id"]: n for n in graph["nary_nodes"]}
    for e in (patch.get("add_entities") or []):
        e.setdefault("aliases", []); ents.setdefault(e.get("id"), e)     # never overwrite an existing card
    for n in (patch.get("add_nary_nodes") or []):
        nodes.setdefault(n.get("id"), n)
    rm = set(patch.get("remove_nary_nodes") or [])
    for nid in rm:
        nodes.pop(nid, None)
    rmrel = {(x.get("subject_id"), x.get("relation_phrase"), x.get("object_id"))
             for x in (patch.get("remove_relations") or [])}
    def _key(r): return (r.get("subject_id"), r.get("relation_phrase") or r.get("property"), r.get("object_id"))
    rels = [r for r in graph["relations"]                                 # drop edges on removed nodes + explicit removals
            if r.get("subject_id") not in rm and r.get("object_id") not in rm and _key(r) not in rmrel]
    rels += (patch.get("add_relations") or [])
    rdefs = graph.get("relation_definitions", []); seen_r = {d.get("relation_phrase") for d in rdefs}
    rdefs += [d for d in (patch.get("add_relation_definitions") or []) if d.get("relation_phrase") not in seen_r]
    tdefs = graph.get("type_definitions", []);     seen_t = {d.get("type") for d in tdefs}
    tdefs += [d for d in (patch.get("add_type_definitions") or []) if d.get("type") not in seen_t]
    graph["entities"] = list(ents.values()); graph["nary_nodes"] = list(nodes.values())
    graph["relations"] = rels; graph["relation_definitions"] = rdefs; graph["type_definitions"] = tdefs
    return graph

def review_graph(aid, text, graph, ref_date=None):
    """SECOND Gemini call: TEXT + draft graph -> PATCH -> apply -> guard -> resolve endpoint texts."""
    before  = (len(graph["entities"]), len(graph["nary_nodes"]), len(graph["relations"]))
    payload = {k: graph[k] for k in ("entities", "nary_nodes", "relations")}
    patch = gemini_json(REVIEW_PROMPT, f"TEXT:\n{text}\n\nDRAFT GRAPH (JSON):\n"
                        + json.dumps(payload, ensure_ascii=False))
    if not isinstance(patch, dict):
        print(f"    [warn] {aid}: review returned no patch object; keeping draft")
        return _resolve_endpoint_texts(graph)
    graph = _apply_patch(graph, patch)
    graph, dropped = _guard_endpoints(graph)
    graph = _resolve_endpoint_texts(graph)
    graph["review_notes"] = patch.get("notes")
    after = (len(graph["entities"]), len(graph["nary_nodes"]), len(graph["relations"]))
    print(f"    review {aid}: entities {before[0]}->{after[0]}  nodes {before[1]}->{after[1]}  rels {before[2]}->{after[2]}  "
          f"(+{len(patch.get('add_entities') or [])} ent, +{len(patch.get('add_relations') or [])} rel, "
          f"collapsed {len(patch.get('remove_nary_nodes') or [])} node, guard-dropped {len(dropped)})")
    return graph


## 2. Inspect the demo article

In [179]:
g = extract_e2e(ARTICLE_ID)
by_id = {e["id"]: e.get("text") for e in g["entities"]}
by_id.update({n["id"]: n["label"] for n in g["nary_nodes"]})
print("-" * 80)
print(f'{len(g["entities"])} entities  |  {len(g["nary_nodes"])} n-ary nodes  |  {len(g["relations"])} relations')
print("-" * 80)
print("n-ary nodes:")
for n in g["nary_nodes"]:
    star = " *MAIN*" if n.get("is_main") else ""
    print(f'  {n["id"]:<5} {n["node_kind"]:<12} <{n.get("type")}>{star}  "{n["label"]}"')
print("-" * 80)
print("relations (first 30):")
for r in g["relations"][:30]:
    s = by_id.get(r["subject_id"], r["subject_id"]); o = by_id.get(r["object_id"], r["object_id"])
    print(f'  {s!r}  --{r["relation_phrase"]}-->  {o!r}')

KeyboardInterrupt: 

## 3. Batch — process the selected articles (`ARTICLE_IDS`)

In [180]:
# Single-call extraction for each article. Re-runs overwrite. Compare against the staged nary_graph.json.
for aid in ARTICLE_IDS:
    print("=" * 60, aid)
    extract_e2e(aid)
print(f"\ndone -- {len(ARTICLE_IDS)} article(s) -> results/<id>/{OUTPUT_NAME}")

============================================================ art1
    review art1: entities 80->80  nodes 33->33  rels 119->125  (+0 ent, +6 rel, collapsed 0 node, guard-dropped 0)
  wrote /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/art1/e2e/nary_graph_dedup.json  (80 entities, 33 n-ary nodes, 125 relations)  [3.1s]
============================================================ art2
    review art2: entities 62->62  nodes 23->23  rels 96->97  (+0 ent, +2 rel, collapsed 0 node, guard-dropped 0)
  wrote /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/art2/e2e/nary_graph_dedup.json  (62 entities, 23 n-ary nodes, 97 relations)  [2.5s]
============================================================ art4
    review art4: entities 47->47  nodes 11->11  rels 53->60  (+0 ent, +8 rel, collapsed 0 node, guard-dropped 1)
  wrote /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/result